# 06 - Evaluation complete (modele 09_spw_neutral) + Generation des labels de soumission
## Hackathon IBM - Detection de Fraude Bancaire

### Ce que fait ce notebook

**Partie A - Evaluation exhaustive du modele 09_spw_neutral sur `prepared_test_050.0_pct.parquet`**
- Toutes les metriques possibles : F1, Precision, Recall, Accuracy, PR-AUC, ROC-AUC, MCC, kappa, balanced accuracy, log-loss, Brier, F-beta (beta=2), specificity, NPV, G-mean...
- Graphiques : ROC, PR, matrice de confusion, calibration, distribution des probas, lift/gain, analyse du seuil, feature importance.

**Partie B - Generation des predictions sur `evaluation_features.csv`**
- Merge evaluation_features + users_data + cards_data + mcc_codes (comme pour le train).
- Applique `preprocess_pipeline.preprocess_dataset(...)`.
- Aligne les categories avec le test set (schema d entrainement).
- Predit via `xgb_09_spw_neutral.ubj` + seuil optimal calcule sur le test.
- Sauve `submission_predictions.csv` (transaction_id, is_fraud, proba).

### Fichiers necessaires dans le repertoire courant
- `logs_training_colab/models/xgb_09_spw_neutral.ubj` (ou chemin equivalent, configurable)
- `prepared_test_050.0_pct.parquet`
- `evaluation_features.csv`
- `users_data.csv`, `cards_data.csv`, `mcc_codes.json` (pour merger)
- `preprocess_pipeline.py`


---
## 0. Imports & configuration


In [ ]:
import os, sys, gc, time, json, warnings
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
                              roc_auc_score, average_precision_score, confusion_matrix,
                              precision_recall_curve, roc_curve, classification_report,
                              matthews_corrcoef, cohen_kappa_score, balanced_accuracy_score,
                              log_loss, brier_score_loss, fbeta_score)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
plt.rcParams['figure.dpi'] = 110

# --- Chemins (ajuste si besoin) ---
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

MODEL_PATH       = ROOT / 'logs_training_colab' / 'models' / 'xgb_09_spw_neutral.ubj'
TEST_PARQUET     = ROOT / 'prepared_test_050.0_pct.parquet'
EVAL_CSV         = ROOT / 'evaluation_features.csv'
USERS_CSV        = ROOT / 'users_data.csv'
CARDS_CSV        = ROOT / 'cards_data.csv'
MCC_JSON         = ROOT / 'mcc_codes.json'
RESULTS_CSV      = ROOT / 'logs_training_colab' / 'experiment_results.csv'  # pour recuperer le best_threshold
OUT_DIR          = ROOT / 'evaluation_outputs_09_spw_neutral'
OUT_DIR.mkdir(exist_ok=True, parents=True)
SUBMISSION_CSV   = OUT_DIR / 'submission_predictions.csv'

# Repli : si le modele n est pas dans logs_training_colab, teste d autres chemins usuels
CANDIDATE_PATHS = [
    MODEL_PATH,
    ROOT / 'logs_training' / 'models' / 'xgb_09_spw_neutral.ubj',
    ROOT / 'xgb_09_spw_neutral.ubj',
]
for p in CANDIDATE_PATHS:
    if p.exists():
        MODEL_PATH = p; break

TARGET = 'is_fraud'

print('Configuration :')
for name, p in [('MODEL', MODEL_PATH), ('TEST', TEST_PARQUET), ('EVAL', EVAL_CSV),
                 ('USERS', USERS_CSV), ('CARDS', CARDS_CSV), ('MCC', MCC_JSON),
                 ('RESULTS', RESULTS_CSV), ('OUT', OUT_DIR)]:
    tag = 'OK  ' if p.exists() else 'MANQUE'
    print(f'  [{tag}] {name:<10} : {p}')


---
## 1. Chargement du modele `xgb_09_spw_neutral`


In [ ]:
assert MODEL_PATH.exists(), f'Modele introuvable : {MODEL_PATH}'

model = xgb.XGBClassifier()
model.load_model(str(MODEL_PATH))
print(f'Modele charge : {MODEL_PATH.name}')

booster = model.get_booster()
feat_names = booster.feature_names
feat_types = booster.feature_types
print(f'  n_features        : {len(feat_names)}')
print(f'  n cat features    : {sum(1 for t in (feat_types or []) if t == "c")}')
print(f'  n total trees     : {booster.num_boosted_rounds()}')
print(f'  best_iteration    : {getattr(model, "best_iteration", "n/a")}')
print(f'  best_ntree_limit  : {getattr(model, "best_ntree_limit", "n/a")}')
print(f'\nFeature names (10 premiers) : {feat_names[:10]}')


---
# Partie A - Evaluation complete sur `prepared_test_050.0_pct.parquet`


## 2. Chargement du test set


In [ ]:
assert TEST_PARQUET.exists(), f'Test set introuvable : {TEST_PARQUET}'
test_df = pd.read_parquet(TEST_PARQUET)
y_test = test_df[TARGET].astype(np.int8).values
X_test = test_df.drop(columns=[TARGET])

# Reordonne les colonnes selon le modele (indispensable)
X_test = X_test[feat_names]

# Les colonnes categorielles doivent etre dtype 'category' pour que enable_categorical marche
cat_cols_model = [n for n, t in zip(feat_names, feat_types or []) if t == 'c']
for c in cat_cols_model:
    if str(X_test[c].dtype) != 'category':
        X_test[c] = X_test[c].astype('category')

print(f'Test set : {X_test.shape}  |  fraudes = {y_test.sum()} / {len(y_test)} ({y_test.mean()*100:.4f}%)')
print(f'Cat cols : {len(cat_cols_model)}')


## 3. Prediction + calcul de TOUTES les metriques


In [ ]:
t0 = time.perf_counter()
y_proba = model.predict_proba(X_test)[:, 1]
print(f'Prediction faite en {time.perf_counter()-t0:.2f}s')

# --- Seuil optimal (maximise F1) via courbe PR ---
prec, rec, thrs = precision_recall_curve(y_test, y_proba)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = int(np.nanargmax(f1s[:-1]))
best_thr = float(thrs[best_idx])
print(f'Seuil optimal (F1 max) : {best_thr:.6f}')

# Optionnel : recupere celui du CSV experiment_results si disponible
if RESULTS_CSV.exists():
    try:
        res = pd.read_csv(RESULTS_CSV)
        row_opt = res[res['Config'] == '09_spw_neutral']
        if len(row_opt) and 'best_threshold' in row_opt.columns:
            thr_csv = float(row_opt['best_threshold'].iloc[0])
            print(f'Seuil dans experiment_results.csv : {thr_csv:.6f}')
    except Exception as e:
        print(f'(info) impossible de lire experiment_results.csv : {e}')

# --- Predictions avec seuil 0.5 et seuil optimal ---
y_pred_05  = (y_proba >= 0.5).astype(int)
y_pred_opt = (y_proba >= best_thr).astype(int)


In [ ]:
def metrics_at(y_true, y_pred, y_proba, tag):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0            # = recall
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0            # specificity
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0            # negative predictive value
    gmean = float(np.sqrt(sens * spec))
    return {
        'tag':              tag,
        'threshold':        0.5 if tag.endswith('0.5') else best_thr,
        'F1':               f1_score(y_true, y_pred, zero_division=0),
        'F2 (beta=2)':      fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        'Precision':        precision_score(y_true, y_pred, zero_division=0),
        'Recall (sens.)':   sens,
        'Specificity':      spec,
        'NPV':              npv,
        'G-Mean':           gmean,
        'Accuracy':         accuracy_score(y_true, y_pred),
        'Balanced Acc':     balanced_accuracy_score(y_true, y_pred),
        'MCC':              matthews_corrcoef(y_true, y_pred),
        'Cohen kappa':      cohen_kappa_score(y_true, y_pred),
        'PR-AUC':           average_precision_score(y_true, y_proba),
        'ROC-AUC':          roc_auc_score(y_true, y_proba),
        'Log loss':         log_loss(y_true, y_proba),
        'Brier':            brier_score_loss(y_true, y_proba),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    }


m05  = metrics_at(y_test, y_pred_05,  y_proba, 'threshold=0.5')
mopt = metrics_at(y_test, y_pred_opt, y_proba, f'threshold={best_thr:.4f} (optimal)')

metrics_df = pd.DataFrame([m05, mopt])
metrics_df = metrics_df[['tag', 'threshold', 'F1', 'F2 (beta=2)', 'Precision',
                          'Recall (sens.)', 'Specificity', 'NPV', 'G-Mean',
                          'Accuracy', 'Balanced Acc', 'MCC', 'Cohen kappa',
                          'PR-AUC', 'ROC-AUC', 'Log loss', 'Brier',
                          'TP', 'FP', 'FN', 'TN']]

pd.options.display.float_format = '{:.6f}'.format
print('=== TABLEAU DE METRIQUES ===\n')
print(metrics_df.to_string(index=False))

metrics_df.to_csv(OUT_DIR / 'metrics_test_set.csv', index=False)
print(f'\nSauve -> {OUT_DIR / "metrics_test_set.csv"}')


In [ ]:
print('=== CLASSIFICATION REPORT (seuil optimal) ===')
print(classification_report(y_test, y_pred_opt, digits=4,
                             target_names=['non_fraud (0)', 'fraud (1)']))


---
## 4. Graphiques d evaluation


In [ ]:
# 4.1 - Matrices de confusion (seuil 0.5 vs optimal)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (y_pred, title) in zip(axes, [(y_pred_05, 'Seuil = 0.5'),
                                         (y_pred_opt, f'Seuil = {best_thr:.4f} (optimal)')]):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['pred non_fraud', 'pred fraud'])
    ax.set_yticklabels(['true non_fraud', 'true fraud'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=12, fontweight='bold')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / 'confusion_matrices.png', bbox_inches='tight')
plt.show()


In [ ]:
# 4.2 - Courbes ROC et Precision-Recall
from sklearn.metrics import auc as sk_auc

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_val = sk_auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(fpr, tpr, lw=2, label=f'XGBoost (ROC-AUC = {roc_auc_val:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Hasard')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('Courbe ROC'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)

ax = axes[1]
pr_auc_val = average_precision_score(y_test, y_proba)
ax.plot(rec, prec, lw=2, label=f'XGBoost (PR-AUC = {pr_auc_val:.4f})')
ax.axhline(y_test.mean(), color='k', linestyle='--', alpha=0.5,
            label=f'Taux de fraude = {y_test.mean():.4f}')
idx_opt = np.argmin(np.abs(thrs - best_thr))
ax.scatter([rec[idx_opt]], [prec[idx_opt]], color='red', s=100, zorder=5,
            label=f'Seuil optimal (thr={best_thr:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Courbe Precision-Recall'); ax.legend(loc='upper right'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'roc_pr_curves.png', bbox_inches='tight')
plt.show()


In [ ]:
# 4.3 - Distribution des probas + analyse du seuil
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bins = np.linspace(0, 1, 61)
ax.hist(y_proba[y_test == 0], bins=bins, alpha=0.6, label='Non-fraude (0)', density=True, color='#4C72B0')
ax.hist(y_proba[y_test == 1], bins=bins, alpha=0.6, label='Fraude (1)',     density=True, color='#C44E52')
ax.axvline(0.5,      color='gray',   linestyle='--', label='Seuil 0.5')
ax.axvline(best_thr, color='green',  linestyle='--', label=f'Seuil optimal = {best_thr:.3f}')
ax.set_xlabel('Probabilite predite (fraude)'); ax.set_ylabel('Densite')
ax.set_title('Distribution des probas'); ax.legend(); ax.grid(alpha=0.3)
ax.set_yscale('log')

ax = axes[1]
thr_grid = np.linspace(0.01, 0.99, 99)
f1_list, prec_list, rec_list = [], [], []
for t in thr_grid:
    yp = (y_proba >= t).astype(int)
    f1_list.append(f1_score(y_test, yp, zero_division=0))
    prec_list.append(precision_score(y_test, yp, zero_division=0))
    rec_list.append(recall_score(y_test, yp, zero_division=0))
ax.plot(thr_grid, f1_list,   label='F1',       lw=2, color='#C44E52')
ax.plot(thr_grid, prec_list, label='Precision', lw=2, color='#4C72B0')
ax.plot(thr_grid, rec_list,  label='Recall',    lw=2, color='#55A868')
ax.axvline(best_thr, color='green', linestyle='--', alpha=0.7)
ax.set_xlabel('Seuil'); ax.set_ylabel('Score')
ax.set_title('Metriques vs seuil'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'threshold_analysis.png', bbox_inches='tight')
plt.show()


In [ ]:
# 4.4 - Calibration curve + cumulative gain / lift chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=15, strategy='quantile')
ax.plot(prob_pred, prob_true, marker='o', lw=2, label='Modele')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Parfaitement calibre')
ax.set_xlabel('Probabilite predite moyenne (par bin)')
ax.set_ylabel('Taux de fraude observe')
ax.set_title('Courbe de calibration (reliability)')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
# Cumulative gains / lift
order = np.argsort(-y_proba)
y_sorted = y_test[order]
cum_gain = np.cumsum(y_sorted) / y_sorted.sum()
percentile = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
ax.plot(percentile, cum_gain, lw=2, label='Modele')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Hasard')
# Au 10 % superieur :
top10_idx = int(len(y_sorted) * 0.10)
gain10 = cum_gain[top10_idx - 1] if top10_idx > 0 else 0
ax.axvline(0.10, color='red', linestyle=':', alpha=0.6)
ax.text(0.11, 0.05, f'Top 10% capture\n{gain10*100:.1f}% des fraudes', fontsize=9)
ax.set_xlabel('Fraction du dataset (classee par proba decroissante)')
ax.set_ylabel('Fraction des fraudes capturees')
ax.set_title('Courbe de gain cumulatif')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'calibration_gains.png', bbox_inches='tight')
plt.show()


In [ ]:
# 4.5 - Feature importance
imp = pd.DataFrame({
    'feature': feat_names,
    'gain':    model.feature_importances_,
}).sort_values('gain', ascending=False)

top_n = min(25, len(imp))
fig, ax = plt.subplots(figsize=(10, max(5, top_n * 0.28)))
imp_top = imp.head(top_n).iloc[::-1]
ax.barh(imp_top['feature'], imp_top['gain'], color='#55A868')
ax.set_xlabel('Importance (gain)')
ax.set_title(f'Top {top_n} feature importances')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_importance.png', bbox_inches='tight')
plt.show()

imp.to_csv(OUT_DIR / 'feature_importance.csv', index=False)
print(f'Sauve -> {OUT_DIR / "feature_importance.csv"}')


---
# Partie B - Generation des labels pour `evaluation_features.csv`


## 5. Chargement et merge des sources brutes


In [ ]:
print('Chargement evaluation_features.csv ...')
eval_raw = pd.read_csv(EVAL_CSV)
print(f'  shape : {eval_raw.shape}')
print(f'  cols  : {list(eval_raw.columns)}')

# Conserve les transaction_id pour la soumission finale
TX_IDS = eval_raw['transaction_id'].copy()
print(f'  transaction_ids uniques : {TX_IDS.nunique()}')

# --- Merge avec users_data ---
print('\nMerge avec users_data.csv ...')
users = pd.read_csv(USERS_CSV)
if 'id' in users.columns:
    users = users.rename(columns={'id': 'client_id'})
df = eval_raw.merge(users, on='client_id', how='left', suffixes=('', '_user'))
print(f'  shape apres users : {df.shape}')

# --- Merge avec cards_data ---
print('\nMerge avec cards_data.csv ...')
cards = pd.read_csv(CARDS_CSV)
if 'id' in cards.columns:
    cards = cards.rename(columns={'id': 'card_id'})
# On drop client_id cote cards pour eviter les conflits (on garde celui des transactions)
cards_cols = [c for c in cards.columns if c != 'client_id']
df = df.merge(cards[cards_cols], on='card_id', how='left', suffixes=('', '_card'))
print(f'  shape apres cards : {df.shape}')

# --- Map MCC ---
print('\nMap mcc_codes.json ...')
with open(MCC_JSON, 'r', encoding='utf-8') as f:
    mcc_map = json.load(f)
# Les cles du JSON sont des strings
df['mcc_description'] = df['mcc'].astype(str).map(mcc_map).fillna('unknown')
print(f'  mcc couverture : {(df["mcc_description"] != "unknown").mean()*100:.2f}%')

print(f'\nDataFrame final avant preprocessing : {df.shape}')
print(f'  memoire : {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')


## 6. Application du preprocessing identique au train


In [ ]:
from preprocess_pipeline import preprocess_dataset

print('Application de preprocess_dataset(...) ...')
t0 = time.perf_counter()
# On ajoute une colonne is_fraud fantome (le pipeline la cherche, mais elle n existe pas ici)
# On utilise 0 pour toutes les lignes ; on droppera la colonne avant prediction.
if TARGET not in df.columns:
    df[TARGET] = 0

df_prep = preprocess_dataset(df, verbose=True)
print(f'\nPreprocess termine en {time.perf_counter()-t0:.1f}s')
print(f'  shape finale : {df_prep.shape}')
print(f'  memoire       : {df_prep.memory_usage(deep=True).sum()/1024**2:.1f} MB')

# On retire la colonne is_fraud fantome
if TARGET in df_prep.columns:
    df_prep = df_prep.drop(columns=[TARGET])


## 7. Alignement du schema avec le modele


In [ ]:
# --- 7.1 Colonnes manquantes / en trop ---
missing_in_eval = [c for c in feat_names if c not in df_prep.columns]
extra_in_eval   = [c for c in df_prep.columns if c not in feat_names]
print(f'Colonnes attendues par le modele       : {len(feat_names)}')
print(f'Colonnes du df_prep apres preprocessing : {df_prep.shape[1]}')
print(f'Manquantes dans eval  (creees a NaN)    : {len(missing_in_eval)} -> {missing_in_eval[:10]}')
print(f'En trop dans eval     (ignorees)        : {len(extra_in_eval)} -> {extra_in_eval[:10]}')

for c in missing_in_eval:
    df_prep[c] = np.nan

# --- 7.2 Recharge le test pour recuperer les niveaux categoriels du train ---
print('\nAlignement des categories avec le test set (reference training) ...')
ref = pd.read_parquet(TEST_PARQUET)

for c in [name for name, t in zip(feat_names, feat_types or []) if t == 'c']:
    if c in ref.columns and str(ref[c].dtype) == 'category':
        ref_cats = ref[c].cat.categories
        df_prep[c] = pd.Categorical(df_prep[c].astype(str).replace('nan', np.nan),
                                     categories=ref_cats.astype(str))
    else:
        if c in df_prep.columns and str(df_prep[c].dtype) != 'category':
            df_prep[c] = df_prep[c].astype('category')

del ref; gc.collect()

# --- 7.3 Reordonne exactement comme le modele ---
X_eval = df_prep[feat_names].copy()
print(f'\nX_eval final : {X_eval.shape}')
print('dtypes (sample):')
print(X_eval.dtypes.value_counts())


## 8. Prediction + sauvegarde de la soumission


In [ ]:
print('Prediction sur evaluation_features...')
t0 = time.perf_counter()
proba_eval = model.predict_proba(X_eval)[:, 1]
print(f'  fait en {time.perf_counter()-t0:.2f}s')

# On utilise le seuil optimal CALCULE SUR LE TEST SET
pred_eval = (proba_eval >= best_thr).astype(int)

print(f'\n=== Distribution predite sur evaluation_features ===')
print(f'  seuil utilise   : {best_thr:.6f}')
print(f'  fraudes predites : {int(pred_eval.sum())} / {len(pred_eval)} ({pred_eval.mean()*100:.3f}%)')
print(f'  proba min/med/max: {proba_eval.min():.4f} / {np.median(proba_eval):.4f} / {proba_eval.max():.4f}')

submission = pd.DataFrame({
    'transaction_id': TX_IDS.values,
    'is_fraud':       pred_eval,
    'probability':    proba_eval.astype(np.float32),
})
submission.to_csv(SUBMISSION_CSV, index=False)
print(f'\nSoumission sauvegardee -> {SUBMISSION_CSV}')
print(f'  ({SUBMISSION_CSV.stat().st_size/1024:.1f} KB)')

# Une 2e version avec seuil 0.5 au cas ou
submission_05 = submission.copy()
submission_05['is_fraud'] = (proba_eval >= 0.5).astype(int)
path_05 = OUT_DIR / 'submission_predictions_thr0.5.csv'
submission_05.to_csv(path_05, index=False)
print(f'Version seuil 0.5  -> {path_05}  ({int((proba_eval >= 0.5).sum())} fraudes predites)')

submission.head(10)


In [ ]:
# Distribution visuelle des probas sur l eval set
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.hist(proba_eval, bins=60, color='#4C72B0', alpha=0.7)
ax.axvline(best_thr, color='green', linestyle='--',
            label=f'Seuil optimal = {best_thr:.4f}  ({int((proba_eval >= best_thr).sum())} fraudes)')
ax.axvline(0.5, color='gray', linestyle='--',
            label=f'Seuil 0.5  ({int((proba_eval >= 0.5).sum())} fraudes)')
ax.set_xlabel('Probabilite predite'); ax.set_ylabel('Nombre de transactions')
ax.set_title('Distribution des probas predites sur evaluation_features.csv')
ax.set_yscale('log')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'proba_dist_eval.png', bbox_inches='tight')
plt.show()


---
## 9. Resume des fichiers produits


In [ ]:
print('=== FICHIERS PRODUITS ===')
for p in sorted(OUT_DIR.glob('*')):
    print(f'  {p.name:<40}  ({p.stat().st_size/1024:.1f} KB)')

print(f'\n=== RESUME EVALUATION TEST SET ===')
print(f'  Seuil optimal             : {best_thr:.6f}')
print(f'  F1 @ optimal              : {mopt["F1"]:.6f}')
print(f'  Recall @ optimal          : {mopt["Recall (sens.)"]:.6f}')
print(f'  Precision @ optimal       : {mopt["Precision"]:.6f}')
print(f'  PR-AUC                    : {mopt["PR-AUC"]:.6f}')
print(f'  ROC-AUC                   : {mopt["ROC-AUC"]:.6f}')
print(f'  MCC                       : {mopt["MCC"]:.6f}')
print(f'  Balanced Accuracy         : {mopt["Balanced Acc"]:.6f}')
print(f'  TP={mopt["TP"]:>6}  FP={mopt["FP"]:>6}  FN={mopt["FN"]:>6}  TN={mopt["TN"]:>6}')

print(f'\n=== SOUMISSION ===')
print(f'  {len(submission)} predictions sur evaluation_features.csv')
print(f'  Fichier principal : {SUBMISSION_CSV}')
